In [ ]:
import numpy as np
import lonboard
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import geopandas as gpd
from pathlib import Path


base = Path().resolve()
while not (base / "readme.md").exists() and base != base.parent:
    base = base.parent
nycha = base / "data/processed/combined_utilities_2024.csv"
nychadf = pd.read_csv(nycha)
nychagdf = gpd.GeoDataFrame(nychadf, geometry=gpd.points_from_xy(nychadf.longitude_nycha, nychadf.latitude_nycha))
nychagdf = nychagdf.set_crs(epsg=4326)

clean_gdf = nychagdf.dropna(subset=["geometry"]).reset_index(drop=True)
clean_gdf = clean_gdf.convert_dtypes(dtype_backend="numpy_nullable")

#Electricity consumption per unit 
clean_gdf["elec"] = pd.to_numeric(
    clean_gdf["electricity_consumption_per_unit"], errors="coerce"
).fillna(0)

cap = clean_gdf["elec"].quantile(0.95)
clean_gdf["elec_capped"] = clean_gdf["elec"].clip(upper=cap)

min_e = clean_gdf["elec_capped"].min()
max_e = clean_gdf["elec_capped"].max()
clean_gdf["elec_norm"] = (clean_gdf["elec_capped"] - min_e) / (max_e - min_e)


def consumption_color(norm_val):
    r = int(33  + (230 - 33)  * norm_val)
    g = int(150 + (57  - 150) * norm_val)
    b = int(243 + (70  - 243) * norm_val)
    return [r, g, b, 200]

colors = np.array([
    consumption_color(v) for v in clean_gdf["elec_norm"]
], dtype=np.uint8)

radii = (4 + clean_gdf["elec_norm"] * 26).to_numpy(dtype=np.float32)
weights = clean_gdf["elec_norm"].to_numpy(dtype=np.float32)


heatmap_layer = lonboard.HeatmapLayer.from_geopandas(
    clean_gdf,
    get_weight=weights,
    radius_pixels=60,
    intensity=1,
    threshold=0.05,
)


scatter_layer = lonboard.ScatterplotLayer.from_geopandas(
    clean_gdf,
    get_fill_color=colors,
    get_radius=radii,
    radius_min_pixels=4,
    radius_max_pixels=30,
    radius_units="pixels",
    stroked=False,
    opacity=0.6,
)


m = lonboard.Map(
    layers=[scatter_layer],
    basemap_style=lonboard.basemap.CartoBasemap.DarkMatter,
)

# legend
legend = widgets.HTML(f"""
<div style="
    background: #1a1a2e;
    border-radius: 10px;
    padding: 16px 20px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.4);
    font-family: Arial, sans-serif;
    font-size: 13px;
    width: 190px;
    color: #eee;
">
  <div style="font-weight:bold; font-size:14px; margin-bottom:4px;">
    Electricity Consumption
  </div>
  <div style="font-size:11px; color:#aaa; margin-bottom:12px;">per unit (kWh)</div>

  <!-- Gradient bar -->
  <div style="
    height: 14px;
    border-radius: 4px;
    background: linear-gradient(to right, rgba(33,150,243,0.9), rgba(230,57,70,0.9));
    margin-bottom: 4px;
  "></div>
  <div style="display:flex; justify-content:space-between; font-size:11px; color:#aaa; margin-bottom:12px;">
    <span>Low</span>
    <span>High</span>
  </div>

  <div style="font-size:11px; color:#aaa; margin-bottom:14px;">
    95th pct cap: <b style="color:#eee;">{cap:,.0f} kWh</b>
  </div>

  <hr style="border:none; border-top:1px solid #333; margin-bottom:12px;">

  <!-- Layers explained -->
  <div style="font-weight:bold; font-size:12px; margin-bottom:8px; color:#ccc;">Layers</div>

  <div style="display:flex; align-items:center; margin-bottom:8px;">
    <div style="
      width:28px; height:14px;
      border-radius:4px;
      background: linear-gradient(to right, rgba(0,200,255,0.1), rgba(255,50,50,0.8));
      margin-right:8px; flex-shrink:0;
    "></div>
    <span style="font-size:11px; color:#aaa;">Heatmap — spatial pattern</span>
  </div>

  <div style="display:flex; align-items:center; margin-bottom:14px;">
    <div style="
      width:14px; height:14px;
      border-radius:50%;
      background:rgba(33,150,243,0.8);
      margin-right:8px; flex-shrink:0;
    "></div>
    <span style="font-size:11px; color:#aaa;">Points — per property</span>
  </div>

  <hr style="border:none; border-top:1px solid #333; margin-bottom:12px;">

  <!-- Size legend -->
  <div style="font-weight:bold; font-size:12px; margin-bottom:8px; color:#ccc;">Point Size</div>
  <div style="display:flex; align-items:center; margin-bottom:6px;">
    <span style="width:8px;height:8px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">Low consumption</span>
  </div>
  <div style="display:flex; align-items:center; margin-bottom:6px;">
    <span style="width:14px;height:14px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">Medium consumption</span>
  </div>
  <div style="display:flex; align-items:center;">
    <span style="width:20px;height:20px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">High consumption</span>
  </div>
</div>
""")

display(m)
display(legend)

/Users/jamie/Library/Python/3.9/lib/python/site-packages/lonboard/_layer.py:2037: UserWarning: 
The `HeatmapLayer` is not currently working.

As of Lonboard v0.10, Lonboard upgraded to version 9.0 of the underlying
[deck.gl](https://deck.gl/) library. deck.gl [appears to have a
bug](https://github.com/visgl/deck.gl/issues/8960#issuecomment-2284791644) with
the HeatmapLayer in 9.0, that has not yet been fixed.

Please temporarily downgrade to Lonboard v0.9 if you would like to use the
`HeatmapLayer`.

  warnings.warn(dedent(err_msg), UserWarning)


HTML(value='\n<div style="\n    background: #1a1a2e;\n    border-radius: 10px;\n    padding: 16px 20px;\n    b…

In [ ]:
clean_gdf["gas"] = pd.to_numeric(
    clean_gdf["gas_consumption_per_unit"], errors="coerce"
).fillna(0)

cap_gas = clean_gdf["gas"].quantile(0.95)
clean_gdf["gas_capped"] = clean_gdf["gas"].clip(upper=cap_gas)

min_g = clean_gdf["gas_capped"].min()
max_g = clean_gdf["gas_capped"].max()
clean_gdf["gas_norm"] = (clean_gdf["gas_capped"] - min_g) / (max_g - min_g)

def gas_color(norm_val):
    # green (low) → orange (high)
    r = int(76  + (255 - 76)  * norm_val)
    g = int(175 + (152 - 175) * norm_val)
    b = int(80  + (0   - 80)  * norm_val)
    return [r, g, b, 200]

colors_gas = np.array([
    gas_color(v) for v in clean_gdf["gas_norm"]
], dtype=np.uint8)

radii_gas = (4 + clean_gdf["gas_norm"] * 26).to_numpy(dtype=np.float32)

scatter_gas = lonboard.ScatterplotLayer.from_geopandas(
    clean_gdf,
    get_fill_color=colors_gas,
    get_radius=radii_gas,
    radius_min_pixels=4,
    radius_max_pixels=30,
    radius_units="pixels",
    stroked=False,
    opacity=0.6,
)

m_gas = lonboard.Map(
    layers=[scatter_gas],
    basemap_style=lonboard.basemap.CartoBasemap.DarkMatter,
)

legend_gas = widgets.HTML(f"""
<div style="
    background: #1a1a2e;
    border-radius: 10px;
    padding: 16px 20px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.4);
    font-family: Arial, sans-serif;
    font-size: 13px;
    width: 190px;
    color: #eee;
">
  <div style="font-weight:bold; font-size:14px; margin-bottom:4px;">Gas Consumption</div>
  <div style="font-size:11px; color:#aaa; margin-bottom:12px;">per unit</div>
  <div style="
    height: 14px;
    border-radius: 4px;
    background: linear-gradient(to right, rgba(76,175,80,0.9), rgba(255,152,0,0.9));
    margin-bottom: 4px;
  "></div>
  <div style="display:flex; justify-content:space-between; font-size:11px; color:#aaa; margin-bottom:12px;">
    <span>Low</span><span>High</span>
  </div>
  <div style="font-size:11px; color:#aaa; margin-bottom:14px;">
    95th pct cap: <b style="color:#eee;">{cap_gas:,.0f}</b>
  </div>
  <hr style="border:none; border-top:1px solid #333; margin-bottom:12px;">
  <div style="font-weight:bold; font-size:12px; margin-bottom:8px; color:#ccc;">Point Size</div>
  <div style="display:flex; align-items:center; margin-bottom:6px;">
    <span style="width:8px;height:8px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">Low consumption</span>
  </div>
  <div style="display:flex; align-items:center; margin-bottom:6px;">
    <span style="width:14px;height:14px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">Medium consumption</span>
  </div>
  <div style="display:flex; align-items:center;">
    <span style="width:20px;height:20px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">High consumption</span>
  </div>
</div>
""")

display(m_gas)
display(legend_gas)

HTML(value='\n<div style="\n    background: #1a1a2e;\n    border-radius: 10px;\n    padding: 16px 20px;\n    b…

In [ ]:
steam_gdf = clean_gdf[clean_gdf["steam"] > 0].reset_index(drop=True)

cap_steam = steam_gdf["steam"].quantile(0.95)
steam_gdf["steam_capped"] = steam_gdf["steam"].clip(upper=cap_steam)

min_s = steam_gdf["steam_capped"].min()
max_s = steam_gdf["steam_capped"].max()
steam_gdf["steam_norm"] = (
    (steam_gdf["steam_capped"] - min_s) / (max_s - min_s)
).fillna(0)

def steam_color(norm_val):
    if np.isnan(norm_val):
        return [80, 80, 80, 100]
    r = int(156 + (255 - 156) * norm_val)
    g = int(39  + (235 - 39)  * norm_val)
    b = int(176 + (59  - 176) * norm_val)
    return [r, g, b, 200]

colors_steam = np.array([
    steam_color(v) for v in steam_gdf["steam_norm"]
], dtype=np.uint8)

radii_steam = (4 + steam_gdf["steam_norm"] * 26).fillna(4).to_numpy(dtype=np.float32)

scatter_steam = lonboard.ScatterplotLayer.from_geopandas(
    steam_gdf,
    get_fill_color=colors_steam,
    get_radius=radii_steam,
    radius_min_pixels=6,
    radius_max_pixels=40,
    radius_units="pixels",
    stroked=False,
    opacity=0.8,
)

m_steam = lonboard.Map(
    layers=[scatter_steam],
    basemap_style=lonboard.basemap.CartoBasemap.DarkMatter,
)

legend_steam = widgets.HTML(f"""
<div style="
    background: #1a1a2e;
    border-radius: 10px;
    padding: 16px 20px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.4);
    font-family: Arial, sans-serif;
    font-size: 13px;
    width: 190px;
    color: #eee;
">
  <div style="font-weight:bold; font-size:14px; margin-bottom:4px;">Steam Consumption</div>
  <div style="font-size:11px; color:#aaa; margin-bottom:12px;">per unit — properties with steam only</div>
  <div style="
    height: 14px;
    border-radius: 4px;
    background: linear-gradient(to right, rgba(156,39,176,0.9), rgba(255,235,59,0.9));
    margin-bottom: 4px;
  "></div>
  <div style="display:flex; justify-content:space-between; font-size:11px; color:#aaa; margin-bottom:12px;">
    <span>Low</span><span>High</span>
  </div>
  <div style="font-size:11px; color:#aaa; margin-bottom:4px;">
    Properties shown: <b style="color:#eee;">21</b>
  </div>
  <div style="font-size:11px; color:#aaa; margin-bottom:14px;">
    95th pct cap: <b style="color:#eee;">{cap_steam:,.0f}</b>
  </div>
  <hr style="border:none; border-top:1px solid #333; margin-bottom:12px;">
  <div style="font-weight:bold; font-size:12px; margin-bottom:8px; color:#ccc;">Point Size</div>
  <div style="display:flex; align-items:center; margin-bottom:6px;">
    <span style="width:8px;height:8px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">Low consumption</span>
  </div>
  <div style="display:flex; align-items:center; margin-bottom:6px;">
    <span style="width:14px;height:14px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">Medium consumption</span>
  </div>
  <div style="display:flex; align-items:center;">
    <span style="width:20px;height:20px;border-radius:50%;background:#888;display:inline-block;margin-right:10px;flex-shrink:0;"></span>
    <span style="font-size:11px; color:#aaa;">High consumption</span>
  </div>
</div>
""")

display(m_steam)
display(legend_steam)

HTML(value='\n<div style="\n    background: #1a1a2e;\n    border-radius: 10px;\n    padding: 16px 20px;\n    b…

In [ ]:
print(f"Zero steam: {(clean_gdf['steam'] == 0).sum()}")
print(f"Non-zero steam: {(clean_gdf['steam'] > 0).sum()}")
print(f"Null steam: {clean_gdf['steam'].isna().sum()}")
print(f"Cap: {cap_steam:,.0f}")
print(clean_gdf['steam'].describe())

Zero steam: 508
Non-zero steam: 21
Null steam: 0
Cap: 0
count             529.0
mean        7532.649273
std        73936.496154
min                 0.0
25%                 0.0
50%                 0.0
75%                 0.0
max      1006138.792353
Name: steam, dtype: Float64
